# 04 - Seleccion y validacion del pipeline final

Objetivo: dejar la entrega cerrada y auditable.

El final de presentacion usa un soft vote simple de tres ramas reales: `separable_headsep`, `globalmel_sep_temporal` y `sep_temporal_f1024`. Este notebook valida el CSV final y muestra por que se selecciono frente al mejor historico expandido.


In [1]:
from pathlib import Path
import hashlib
import json
import sys

import pandas as pd
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

ROOT = Path.cwd()
if ROOT.name == '04_final':
    ROOT = ROOT.parent
INVESTIGATION = ROOT / 'investigation'
sys.path.insert(0, str(INVESTIGATION))

from scripts.fat2019.submission import label_columns_from_sample, read_sample_submission, validate_submission

DATA_DIR = ROOT / 'data'
sample = read_sample_submission(DATA_DIR / 'sample_submission.csv')
label_columns = label_columns_from_sample(sample)
decision_matrix = pd.read_csv(ROOT / '03_entrenamiento' / 'decision_matrix.csv')
manifest = pd.read_csv(ROOT / '04_final' / 'final_pipeline_manifest.csv')
submission_candidates = pd.read_csv(ROOT / '04_final' / 'submission_candidates.csv')
metadata = json.loads((ROOT / '04_final' / 'final_pipeline_metadata.json').read_text())

print(f'rows expected: {len(sample)}')
print(f'label columns: {len(label_columns)}')
print(metadata['final_submission'])


rows expected: 3361
label columns: 80
{'path': '04_final/submission.csv', 'source': 'investigation/results/submissions/simple_headsep_globalmel_f1024_equal.csv', 'downloaded_equivalent': None, 'sha256': '81ce2b49e836ca89b27e07b2f281eebce3efc103d223c29aa6a3731b7659be9b', 'rows': 3361, 'columns': 81, 'format_reference': 'data/sample_submission.csv'}


## 1. Evidencia de seleccion

El criterio ya no es maximizar el private LB a cualquier costo. Se elige el mejor CSV validado que sea defendible en presentacion: pocas ramas reales, sin esconder el ensemble `current`, formato correcto, SHA registrado y score Kaggle verificable.


In [2]:
candidate_table = submission_candidates.copy()
candidate_table['private_score_num'] = pd.to_numeric(candidate_table['private_score'], errors='coerce')
candidate_table = candidate_table.sort_values(['decision', 'private_score_num'], ascending=[True, False])
cols = ['candidate', 'artifact', 'private_score', 'decision', 'kaggle_description', 'notes']
display(candidate_table[cols])

selected = candidate_table[candidate_table['decision'].eq('selected_final')][cols]
selected


,candidate,artifact,private_score,decision,kaggle_description,notes
4,current85_sep_temporal_full15,investigation/results/submissions/current85_se...,NaN,candidate_needs_kaggle_notebook,NaN,Holdout analog improves 0.841179 to 0.846742; ...
5,current835_sep_temporal_full165,investigation/results/submissions/current835_s...,NaN,candidate_needs_kaggle_notebook,NaN,Geron ensemble sweep; local 0.846913 with arit...
6,current795_sep_temporal_rowz205,investigation/results/submissions/current795_s...,NaN,candidate_needs_kaggle_notebook,NaN,Row-wise z-score blend; local 0.847759
7,current_rowz205_rowrank190_avg,investigation/results/submissions/current_rowz...,NaN,candidate_needs_kaggle_notebook,NaN,Best local Geron candidate; row_z 20.5 plus ro...
11,current475_globalmel200_se125_f1024_200,investigation/results/submissions/current475_g...,0.67025,historical_best_expanded,current475 globalmel200 se125 f1024_200,"Best verified Kaggle private score, but expand..."
2,near_local,investigation/submissions/catsdogs_headsep_fin...,NaN,not_selected,NaN,No direct Kaggle score mapping found in submis...
3,replace_sep_h15,investigation/submissions/catsdogs_headsep_fin...,NaN,not_selected,NaN,No direct Kaggle score mapping found in submis...
12,simple_headsep_globalmel_f1024_equal,investigation/results/submissions/simple_heads...,0.66649,selected_final,simple current-free headsep globalmel f1024 equal,Selected presentation final: 3 real components...
0,translated_local,investigation/submissions/catsdogs_headsep_fin...,0.65289,superseded_verified,catsdogs headsep translated local verified,Matches investigation/kaggle_dataset_download_...
1,conservative_h10,investigation/submissions/catsdogs_headsep_fin...,0.65061,valid_backup,catsdogs headsep conservative h10 verified,Matches investigation/kaggle_dataset_download_...


,candidate,artifact,private_score,decision,kaggle_description,notes
12,simple_headsep_globalmel_f1024_equal,investigation/results/submissions/simple_heads...,0.66649,selected_final,simple current-free headsep globalmel f1024 equal,Selected presentation final: 3 real components...


## 2. Formula del blend final

La submission seleccionada usa tres ramas reales y pesos iguales. La interpretacion importante es diversidad: `separable_headsep` aporta una CNN separable-residual fuerte, `globalmel` cambia la normalizacion de entrada y `f1024` agrega mayor contexto temporal. El candidato historico de `0.67025` queda como referencia, pero no como final de presentacion porque se expande a un blend anidado de 10 componentes reales.


In [3]:
final_blend = pd.DataFrame([
    {'component': 'separable_headsep', 'weight': 1 / 3, 'role': 'CNN separable-residual fuerte'},
    {'component': 'globalmel_sep_temporal', 'weight': 1 / 3, 'role': 'normalizacion global por banda mel'},
    {'component': 'sep_temporal_f1024', 'weight': 1 / 3, 'role': 'mayor contexto temporal'},
])
final_blend.assign(weight_percent=(final_blend['weight'] * 100).round(2))


,component,weight,role,weight_percent
0,separable_headsep,0.333333,CNN separable-residual fuerte,33.33
1,globalmel_sep_temporal,0.333333,normalizacion global por banda mel,33.33
2,sep_temporal_f1024,0.333333,mayor contexto temporal,33.33


## 3. Validacion del archivo final

Se valida contra `sample_submission.csv`: mismas columnas, misma cantidad de filas y probabilidades en `[0, 1]`. Tambien se verifica que el SHA coincida con el candidato seleccionado.


In [4]:
final_path = ROOT / metadata['final_submission']['path']
source_path = ROOT / metadata['final_submission']['source']

submission = pd.read_csv(final_path)
validate_submission(submission, label_columns, expected_rows=len(sample))

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

validation = pd.DataFrame([
    {'check': 'final exists', 'value': final_path.exists()},
    {'check': 'source exists', 'value': source_path.exists()},
    {'check': 'same sha final/source', 'value': sha256(final_path) == sha256(source_path)},
    {'check': 'sha256', 'value': sha256(final_path)},
    {'check': 'rows', 'value': len(submission)},
    {'check': 'columns', 'value': len(submission.columns)},
    {'check': 'min_probability', 'value': submission[label_columns].min().min()},
    {'check': 'max_probability', 'value': submission[label_columns].max().max()},
])
validation


,check,value
0,final exists,True
1,source exists,True
2,same sha final/source,True
3,sha256,81ce2b49e836ca89b27e07b2f281eebce3efc103d223c2...
4,rows,3361
5,columns,81
6,min_probability,0.0
7,max_probability,0.99998


In [5]:
print(metadata['kaggle_evidence'])
submission.head()


{'competition': 'freesound-audio-tagging-2019', 'query': 'kaggle competitions submissions -c freesound-audio-tagging-2019', 'queried_at': '2026-07-01', 'description': 'simple current-free headsep globalmel f1024 equal', 'submission_date': '2026-07-01 19:13:14', 'public_score': 0.0, 'private_score': 0.66649, 'note': 'Public score is 0.00000 for these post-competition submissions in the Kaggle API output; selection uses privateScore.'}


,fname,Accelerating_and_revving_and_vroom,Accordion,Acoustic_guitar,Applause,Bark,Bass_drum,Bass_guitar,Bathtub_(filling_or_washing),Bicycle_bell,...,Toilet_flush,Traffic_noise_and_roadway_noise,Trickle_and_dribble,Walk_and_footsteps,Water_tap_and_faucet,Waves_and_surf,Whispering,Writing,Yell,Zipper_(clothing)
0,4260ebea.wav,0.000026,0.000004,0.000025,0.000012,0.000061,0.000008,0.000104,0.355299,0.000368,...,0.000180,0.000002,0.012407,0.000334,0.458163,0.000240,0.000429,4.301374e-03,0.000010,0.000253
1,426eb1e0.wav,0.000054,0.000062,0.000001,0.997249,0.000136,0.000008,0.000004,0.007598,0.000039,...,0.000117,0.001443,0.002243,0.013392,0.000145,0.000954,0.000013,4.566998e-07,0.001183,0.000248
2,428d70bb.wav,0.000265,0.000192,0.000056,0.000002,0.001657,0.000100,0.000136,0.000382,0.000882,...,0.000280,0.004247,0.000034,0.000104,0.000025,0.000361,0.014325,4.572132e-03,0.003263,0.003125
3,4292b1c9.wav,0.007690,0.000040,0.000067,0.000001,0.000018,0.000006,0.000030,0.000035,0.000052,...,0.000136,0.000026,0.006346,0.000003,0.000668,0.000002,0.000109,2.770795e-05,0.001275,0.003543
4,429c5071.wav,0.001870,0.000054,0.000393,0.000028,0.019817,0.012616,0.000620,0.000380,0.000107,...,0.000060,0.000020,0.000010,0.083285,0.000056,0.003923,0.000163,7.551544e-03,0.000115,0.006882


## 4. Manifest de artefactos

El manifest lista que archivos sostienen la entrega. Si algo falta, no deberia entregarse.


In [6]:
manifest_checks = []
for row in manifest.itertuples(index=False):
    artifact = ROOT / row.artifact
    manifest_checks.append({
        'component': row.component,
        'role': row.role,
        'status': row.status,
        'artifact': row.artifact,
        'exists': artifact.exists(),
    })
manifest_checks = pd.DataFrame(manifest_checks)
manifest_checks


,component,role,status,artifact,exists
0,sklearn_logmel_c001,tabular_diversity,historical_supporting,investigation/models/variants_regsearch_lowc,True
1,head256_relu_cnn,main_neural_branch,historical_supporting,investigation/models/logmel_cnn_fashion_head25...,True
2,separable_headsep,architectural_diversity,selected_final_component_submission,investigation/submissions/logmel_cnn_catsdogs_...,True
3,resnet50_frozen,transfer_diversity,historical_supporting,investigation/models/imagenet_transfer_catsdog...,True
4,headsep_conservative_submission,validated_candidate_submission,validated_candidate,investigation/kaggle_dataset_download_headsep_...,True
5,headsep_translated_submission,validated_candidate_submission,validated_candidate,investigation/kaggle_dataset_download_headsep_...,True
6,final_submission,selected_final_submission,selected_final_submission,04_final/submission.csv,True
7,globalmel_sep_temporal,normalization_component,selected_final_component_submission,investigation/submissions/theory_globalmel_sep...,True
8,sep_temporal_f1024,temporal_context_component,selected_final_component_submission,investigation/submissions/theory_sep_temporal_...,True
9,temporal_bigru,discarded_experiment,not_selected,investigation/results/logmel_cnn_temporal_bigr...,True


## 5. Que no se debe confundir

- El score `0.67025` es el mejor historico expandido, no la seleccion final de presentacion.
- El score `0.66649` es la seleccion actual: 3 componentes reales, sin `current`.
- El public score aparece `0.00000` en la API para estas submissions post-competencia; se compara por private LB.
- No se toca el Kaggle del curso; todo esto refiere a `freesound-audio-tagging-2019`.
- Los candidatos con buena validacion local pero peor Kaggle quedan como evidencia, no como seleccion final.


In [7]:
selected_decisions = decision_matrix[decision_matrix['decision'].isin(['keep', 'blend-only', 'reference'])]
selected_decisions[['idea', 'decision', 'project_use', 'caveat']]


,idea,decision,project_use,caveat
0,priors,reference,Smoke test de formato,No cuenta como modelo acustico
1,sklearn_logmel_stats,keep,Baseline tabular defendible,Colapsa eje temporal
2,sklearn_logmel_c001,keep,Baseline tabular fuerte y rama de blend,No seguir micro-ajustando C si no mejora LB
4,cnn_logmel_image,keep,Modelo neural base,Mas costoso y requiere cache log-mel image
5,cnn_sklearn_blend,keep,Primer blend defendible,Pesos deben documentarse como empiricos si vie...
6,full_train_two_seed_cnn,keep,Entrenamiento final de configuracion elegida,No agregar seeds indefinidamente
7,fashion_head256_training,keep,Rama neural principal,Explicar como paquete de entrenamiento/cabeza ...
9,separable_residual_headsep,keep,Rama final si artefactos estan validados,Incremento final marginal
10,resnet50_imagenet_frozen,blend-only,Rama diversa de ensemble,No presentarlo como mejor modelo individual
13,logmel_cnn_separable_temporal_bigru,blend-only,Candidato de mejora para blend,No reemplazar la final Kaggle-verificada hasta...


## Checklist final

- `04_final/submission.csv` existe y valida contra `sample_submission.csv`.
- El SHA coincide con `simple_headsep_globalmel_f1024_equal.csv`.
- La seleccion final y las alternativas estan documentadas en `04_final/submission_candidates.csv`.
- Las decisiones tecnicas estan justificadas en `decisiones_config_y_proceso.md` y `04_final/final_selection.md`.
- Si aparece algo nuevo en `investigation/results/`, se compara contra `0.66649` bajo la restriccion de 3-4 componentes reales; `0.67025` queda como referencia historica expandida.
